# 02 · Jersey identity: what was read, and where it fails

The blog's two identity metrics, computed on your run:
* players with a known number **at serve time** (all 12 on court + libero, every serve)
* touches with a number

plus the breakdown by court side, which shows the weak spot: the team facing the camera rarely shows its back.

In [ ]:
import os, sys, json, time
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))
os.chdir(REPO)
print("working in", REPO)

In [ ]:
from vb.config import load_config
from vb.pipeline import run_offline
import pandas as pd, numpy as np
cfg = load_config("configs/default.yaml")
cfg.video.source = "s3://YOUR-BUCKET/volleyball/clip.mp4"
cfg.court.seed_pixels, cfg.court.seed_court = [], []     # <- paste from notebook 01
cfg.output.render = False
res = run_offline(cfg, run_name="identity")

In [ ]:
s = res.summary()
print("players with a known number at serve time:", s["numbers_at_serve"])
print("touches with a number:", s["touches_with_number"], "/", s["touches_in_rallies"])

In [ ]:
tt = [t for r in res.rallies for t in r.touches]
by_side = pd.DataFrame([{"side": "near" if t.side == 0 else "far", "has_number": t.number is not None} for t in tt])
by_side.groupby("side").has_number.agg(["sum", "count", "mean"])

In [ ]:
pd.DataFrame(res.slots).sort_values(['side', 'slot'])